In [40]:
import os
from google import genai
from google.genai import types
from tavily import TavilyClient
GEMINI_API_KEY = os.getenv('GEMINI_TOKEN')
GEMINI_MODEL = 'gemini-2.5-flash'
# Only run this block for Gemini Developer API
client = genai.Client(api_key=GEMINI_API_KEY)

In [41]:
prompt = 'Ciao!'

In [7]:
parts = [
        types.Part.from_text(text=prompt),
    ]
content_list = [
        types.Content(
            role='user',
            parts=parts
        )
]
res = client.models.generate_content(
            model=GEMINI_MODEL,
            contents=content_list,
            config=types.GenerateContentConfig(
                temperature=0.3,
            )
    )
res.text

'Ciao! How can I help you today?'

In [42]:
class Agent:
    def __init__(self, system="", tools = dict()):
        self.system = system
        self.messages = []
        self.tools = tools

    def __call__(self, message):
        self.messages.append(
        types.Content(
            role='user',
            parts=[
        types.Part.from_text(text=message),
    ]
        ))
        result = self.execute()
        self.messages.append(types.Content(
            role='model',
            parts=[
        types.Part.from_text(text=result),
    ]
        ))
        return result

    def execute(self):
        
        while True:
            res = client.models.generate_content(
            model=GEMINI_MODEL,
            
            contents=self.messages,
            config=types.GenerateContentConfig(
                system_instruction=self.system,
                temperature=0.3,
                tools = list(self.tools.values()),
                automatic_function_calling=types.AutomaticFunctionCallingConfig(
                    disable=True
                )
            )
            )
            if res.function_calls:
                for call in res.function_calls:
                    try:
                        function_result = self.tools[call.name](
                            **call.args
                        )
                        function_response = {'result': function_result}
                    except Exception as e:
                        function_response = {'error': str(e)}
                function_response_part = types.Part.from_function_response(
                name=call.name,
                response=function_response,
            )
            
                ##IMPORTANTE: DOBBIAMO FAR CAPIRE AL MOEDLLO CHE LA RISPOSTA è UNA TOOL CALL QUINDI IL ROLE DEVE ESSERE SETTATO A tool
                function_response_content = types.Content(
                    role='tool', parts=[function_response_part]
                )
                self.messages.append(function_response_content)
            else:
                break
        
        return res.text

In [43]:
def tavily_search_tool(
    query: str, max_results: int = 5
) -> list[dict]:
    """
    Perform a search using the Tavily API.

    Args:
        query (str): The search query.
        max_results (int): Number of results to return (default 5).
        include_images (bool): Whether to include image results.

    Returns:
        List[dict]: A list of dictionaries with keys like 'title', 'content', and 'url'.
    """
    api_key = os.getenv("TAVILY_API_KEY")
    if not api_key:
        raise ValueError("TAVILY_API_KEY not found in environment variables.")

    client = TavilyClient(api_key)

    try:
        response = client.search(
            query=query, max_results=max_results
        )

        results = []
        for r in response.get("results", []):
            results.append(
                {
                    "title": r.get("title", ""),
                    "content": r.get("content", ""),
                    "url": r.get("url", ""),
                }
            )

        return results

    except Exception as e:
        return [{"error": str(e)}]  # For LLM-friendly agents

In [44]:
bot = Agent(system = 'Sei un assistente AI che ha a disposizione dei tool per cercare informazioni online per rispondere alle richieste dell utente.', tools = {"tavily_search_tool": tavily_search_tool})

In [45]:
bot('Mi dici i principali eventi di Roma di oggi 11 04 2026?')

"Certamente! Ecco i principali eventi a Roma per l'11 aprile 2026:\n\n*   **Rendez-Vous: Festival del Nuovo Cinema Francese** (ultimo giorno, iniziato il 7 aprile).\n*   **Mostra personale di Martina Zanin: Every caress, a blow** (fino al 18 aprile).\n*   **L'ultimo Matisse – Morfologie di carta** (esposte alcune opere).\n*   **Vintage Market** presso San Paolo District (anche il 12 aprile).\n*   **FrankenBierFest** alla Limonaia di Villa Torlonia (fino al 12 aprile).\n*   **Romics**, Festival Internazionale del Fumetto, Animazione, Cinema e Games, alla Fiera di Roma (fino al 12 aprile).\n*   **Giornata dedicata all'Argentina** con workshop, masterclass e degustazione di prodotti tipici presso il Mercato Centrale Roma.\n*   **Festival Contemporaneo Futuro** (fino al 12 aprile).\n*   **Sagra del Carciofo Romanesco di Ladispoli** (fino al 12 aprile).\n*   **Una serata divina** allo Sport City."

In [47]:
bot('Adesso mi cerchi più informazioni per ognuno di quegli eventi?')

'Ecco le informazioni dettagliate per ciascuno degli eventi a Roma per l\'11 aprile 2026, basate sulle ricerche effettuate:\n\n*   **Rendez-Vous: Festival del Nuovo Cinema Francese**:\n    *   **Date**: Dal 7 all\'11 aprile 2026 (alcune fonti indicano fino al 15 aprile 2026).\n    *   **Descrizione**: È la XVI edizione del festival che porta il meglio del cinema francese contemporaneo nelle sale italiane. L\'11 aprile è l\'ultimo giorno (o uno degli ultimi giorni) dell\'evento.\n\n*   **Mostra personale di Martina Zanin: Every caress, a blow**:\n    *   **Date**: Dal 19 febbraio al 18 aprile 2026.\n    *   **Descrizione**: Si tratta di una mostra personale di Martina Zanin. Non sono disponibili ulteriori dettagli specifici sull\'esposizione.\n\n*   **L\'ultimo Matisse – Morfologie di carta**:\n    *   **Descrizione**: Vengono esposte alcune opere di Matisse. Non sono disponibili ulteriori dettagli specifici sull\'esposizione o sulla location.\n\n*   **Vintage Market presso San Paolo Di

In [48]:
bot.messages

[Content(
   parts=[
     Part(
       text='Mi dici i principali eventi di Roma di oggi 11 04 2026?'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       function_response=FunctionResponse(
         name='tavily_search_tool',
         response={
           'result': [
             {<... 3 items at Max depth ...>},
             {<... 3 items at Max depth ...>},
             {<... 3 items at Max depth ...>},
             {<... 3 items at Max depth ...>},
             {<... 3 items at Max depth ...>},
           ]
         }
       )
     ),
   ],
   role='tool'
 ),
 Content(
   parts=[
     Part(
       text="""Certamente! Ecco i principali eventi a Roma per l'11 aprile 2026:
 
 *   **Rendez-Vous: Festival del Nuovo Cinema Francese** (ultimo giorno, iniziato il 7 aprile).
 *   **Mostra personale di Martina Zanin: Every caress, a blow** (fino al 18 aprile).
 *   **L'ultimo Matisse – Morfologie di carta** (esposte alcune opere).
 *   **Vintage Market** presso San Paolo 